In [1]:
# =========================================================
# STEP 5 — SLA PERFORMANCE ANALYTICS
# + AI STRATEGIC INSIGHT ENGINE
# FILE: notebooks/04_sla_performance_analytics.ipynb
# PROJECT: Smart Complaint Analytics System
# INPUT : data/processed/clean_complaints.csv
# OUTPUT: data/processed/executive_summary.json (optional)
# =========================================================

# =========================================================
# 5.1 IMPORT LIBRARIES
# =========================================================

import warnings
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

warnings.filterwarnings("ignore")

# =========================================================
# 5.2 ELITE DARK THEME CONFIGURATION
# =========================================================

DARK_BG = "#061226"
PAPER_BG = "#081a33"
GRID_COLOR = "rgba(255,255,255,0.10)"
FONT_COLOR = "#E8F1FF"

pio.templates.default = "plotly_dark"

COMMON_LAYOUT = dict(
    paper_bgcolor=PAPER_BG,
    plot_bgcolor=DARK_BG,
    font=dict(
        family="Inter, Segoe UI, sans-serif",
        color=FONT_COLOR,
        size=14
    ),
    title=dict(
        font=dict(size=24, color="white"),
        x=0.02,
        xanchor="left"
    ),
    legend=dict(
        bgcolor="rgba(0,0,0,0)",
        borderwidth=0,
        font=dict(color=FONT_COLOR)
    ),
    margin=dict(l=60, r=40, t=80, b=60)
)

In [2]:
# =========================================================
# 5.3 LOAD CLEAN DATASET
# =========================================================

DATA_PATH = Path("../data/processed/clean_complaints.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["created_at"])

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (10000, 25)


,complaint_id,created_at,channel,citizen_comment,complaint_category,department,district,village,sentiment,urgency_level,...,engagement_score,year,month,month_name,quarter,day_of_week,is_resolved,sla_met_flag,is_critical,priority_bucket
0,CMP-04708,2024-01-01 03:40:36,Call Center,Tempat pembuangan di Dadaprejo sudah penuh.,Waste Management,Dinas Lingkungan Hidup,Junrejo,Dadaprejo,Negative,Medium,...,3877,2024,1,January,1,Monday,1,0,0,Medium
1,CMP-07774,2024-01-01 03:43:25,Call Center,Aspal rusak parah di daerah Temas dan belum di...,Road Infrastructure,Dinas PUPR,Batu,Temas,Negative,Low,...,845,2024,1,January,1,Monday,1,0,0,Low
2,CMP-01189,2024-01-01 06:35:30,LAPOR!,Air PDAM di Mojorejo keruh dan tidak layak dig...,Clean Water,PDAM,Bumiaji,Mojorejo,Neutral,Medium,...,1187,2024,1,January,1,Monday,0,0,0,Medium
3,CMP-01853,2024-01-01 10:09:18,Website,Genangan air sering terjadi di Dadaprejo saat ...,Flooding,Dinas PUPR,Junrejo,Dadaprejo,Neutral,High,...,4440,2024,1,January,1,Monday,0,0,0,High
4,CMP-07824,2024-01-01 14:43:25,TikTok,Lampu jalan rusak di kawasan Gunungsari.,Street Lighting,Dinas Perhubungan,Junrejo,Gunungsari,Negative,Medium,...,1780,2024,1,January,1,Monday,1,1,0,Medium


In [3]:
# =========================================================
# 5.4 EXECUTIVE KPI CALCULATION
# =========================================================

total_complaints = len(df)
resolution_rate = df["is_resolved"].mean() * 100
sla_compliance = df["sla_met_flag"].mean() * 100
critical_rate = df["is_critical"].mean() * 100
avg_response_time = df["response_time_hours"].mean()
avg_resolution_days = df["resolution_time_days"].dropna().mean()

top_category = df["complaint_category"].value_counts().idxmax()
top_department = df["department"].value_counts().idxmax()

print("=" * 60)
print("EXECUTIVE KPI SUMMARY")
print("=" * 60)
print(f"Total Complaints        : {total_complaints:,}")
print(f"Resolution Rate         : {resolution_rate:.2f}%")
print(f"SLA Compliance          : {sla_compliance:.2f}%")
print(f"Critical Complaint Rate : {critical_rate:.2f}%")
print(f"Avg Response Time       : {avg_response_time:.2f} hours")
print(f"Avg Resolution Time     : {avg_resolution_days:.2f} days")
print(f"Top Category            : {top_category}")
print(f"Top Department          : {top_department}")

EXECUTIVE KPI SUMMARY
Total Complaints        : 10,000
Resolution Rate         : 47.79%
SLA Compliance          : 38.68%
Critical Complaint Rate : 8.01%
Avg Response Time       : 35.73 hours
Avg Resolution Time     : 6.39 days
Top Category            : Road Infrastructure
Top Department          : Dinas PUPR


In [4]:
# =========================================================
# 5.5 DEPARTMENT EFFICIENCY INDEX
# =========================================================

department_kpi = (
    df.groupby("department")
      .agg(
          total_complaints=("complaint_id", "count"),
          resolution_rate=("is_resolved", "mean"),
          sla_compliance=("sla_met_flag", "mean"),
          avg_response_time=("response_time_hours", "mean")
      )
      .reset_index()
)

department_kpi["resolution_rate"] *= 100
department_kpi["sla_compliance"] *= 100

# Composite score (0–100)
department_kpi["efficiency_index"] = (
    department_kpi["resolution_rate"] * 0.4
    + department_kpi["sla_compliance"] * 0.4
    + (100 - department_kpi["avg_response_time"].clip(upper=100)) * 0.2
)

department_kpi = department_kpi.sort_values(
    "efficiency_index",
    ascending=False
)

department_kpi.head()

,department,total_complaints,resolution_rate,sla_compliance,avg_response_time,efficiency_index
5,Dinas Perhubungan,1945,48.226221,40.205656,35.977810,48.177189
2,Dinas Lingkungan Hidup,1000,47.700000,40.000000,35.430810,47.993838
1,Dinas Kependudukan,983,46.286877,41.200407,35.252269,47.944460
4,Dinas Pariwisata,1026,46.101365,40.155945,35.227027,47.457519
0,DISKOMINFO,971,49.639547,36.251287,35.413378,47.273658


In [5]:
# =========================================================
# 5.6 DEPARTMENT EFFICIENCY VISUAL (DARK THEME)
# =========================================================

fig = px.bar(
    department_kpi,
    x="efficiency_index",
    y="department",
    orientation="h",
    title="🏛️ Department Efficiency Index",
    template="plotly_dark",
    color="efficiency_index",
    color_continuous_scale="Viridis"
)

fig.update_layout(
    **COMMON_LAYOUT,
    xaxis_title="Efficiency Index",
    yaxis_title="Department",
    yaxis=dict(categoryorder="total ascending")
)

fig.show()

# Save screenshot as:
# assets/department-efficiency-index.png

In [6]:
# =========================================================
# 5.7 COMPLAINT RISK MATRIX
# =========================================================

risk_matrix = (
    df.groupby("complaint_category")
      .agg(
          avg_priority=("priority_score", "mean"),
          total_complaints=("complaint_id", "count")
      )
      .reset_index()
)

fig = px.scatter(
    risk_matrix,
    x="total_complaints",
    y="avg_priority",
    size="total_complaints",
    color="avg_priority",
    hover_name="complaint_category",
    title="⚠️ Complaint Risk Matrix",
    template="plotly_dark",
    color_continuous_scale="Turbo"
)

fig.update_layout(
    **COMMON_LAYOUT,
    xaxis_title="Complaint Volume",
    yaxis_title="Average Priority Score"
)

fig.show()

# Save screenshot as:
# assets/complaint-risk-matrix.png

In [7]:
# =========================================================
# 5.8 AUTOMATED STRATEGIC INSIGHT ENGINE
# =========================================================

recommendations = []

if resolution_rate < 75:
    recommendations.append(
        "Increase operational capacity to improve complaint resolution rates."
    )

if sla_compliance < 80:
    recommendations.append(
        "Strengthen SLA monitoring and response escalation procedures."
    )

if critical_rate > 10:
    recommendations.append(
        "Establish a rapid response team for high-priority complaints."
    )

if avg_response_time > 24:
    recommendations.append(
        "Optimize internal workflows to reduce response time."
    )

if not recommendations:
    recommendations.append(
        "Operational performance is strong and currently within target."
    )

for i, rec in enumerate(recommendations, start=1):
    print(f"{i}. {rec}")

1. Increase operational capacity to improve complaint resolution rates.
2. Strengthen SLA monitoring and response escalation procedures.
3. Optimize internal workflows to reduce response time.


In [8]:
# =========================================================
# 5.9 GOVERNMENT RESPONSE STATUS
# =========================================================

if sla_compliance >= 90 and resolution_rate >= 85:
    response_status = "Excellent"
elif sla_compliance >= 80 and resolution_rate >= 75:
    response_status = "Strong"
elif sla_compliance >= 65 and resolution_rate >= 60:
    response_status = "Needs Improvement"
else:
    response_status = "Critical Attention Required"

print("Government Response Status:", response_status)

Government Response Status: Critical Attention Required


In [9]:
# =========================================================
# 5.10 EXPORT EXECUTIVE SUMMARY (OPTIONAL)
# =========================================================

summary = {
    "total_complaints": int(total_complaints),
    "resolution_rate": round(resolution_rate, 2),
    "sla_compliance": round(sla_compliance, 2),
    "critical_complaint_rate": round(critical_rate, 2),
    "average_response_time_hours": round(avg_response_time, 2),
    "average_resolution_time_days": round(avg_resolution_days, 2),
    "top_complaint_category": top_category,
    "top_department": top_department,
    "government_response_status": response_status,
    "recommendations": recommendations
}

OUTPUT_PATH = Path("../data/processed/executive_summary.json")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

print(f"Executive summary saved to: {OUTPUT_PATH}")

Executive summary saved to: ..\data\processed\executive_summary.json


In [10]:
# =========================================================
# 5.11 EXECUTIVE SUMMARY REPORT
# =========================================================

print(f"""
============================================================
SMART COMPLAINT ANALYTICS SYSTEM
AI STRATEGIC INSIGHT EXECUTIVE SUMMARY
============================================================

Total Complaints        : {total_complaints:,}
Resolution Rate         : {resolution_rate:.2f}%
SLA Compliance          : {sla_compliance:.2f}%
Critical Complaint Rate : {critical_rate:.2f}%
Average Response Time   : {avg_response_time:.2f} hours
Average Resolution Time : {avg_resolution_days:.2f} days

Top Complaint Category  : {top_category}
Top Department          : {top_department}

Government Response Status:
{response_status}

Strategic Recommendations:
{chr(10).join([f"- {r}" for r in recommendations])}

============================================================
""")


SMART COMPLAINT ANALYTICS SYSTEM
AI STRATEGIC INSIGHT EXECUTIVE SUMMARY

Total Complaints        : 10,000
Resolution Rate         : 47.79%
SLA Compliance          : 38.68%
Critical Complaint Rate : 8.01%
Average Response Time   : 35.73 hours
Average Resolution Time : 6.39 days

Top Complaint Category  : Road Infrastructure
Top Department          : Dinas PUPR

Government Response Status:
Critical Attention Required

Strategic Recommendations:
- Increase operational capacity to improve complaint resolution rates.
- Strengthen SLA monitoring and response escalation procedures.
- Optimize internal workflows to reduce response time.


